# Quantum-Guided Cluster Algorithm for Max-Cut

This notebook implements the **Quantum-Guided Cluster Algorithm (QGCA)** from [arXiv:2508.10656](https://arxiv.org/abs/2508.10656), using the `divi` quantum programming framework.

**The idea:** Run QAOA *once* to extract two-point spin correlations ⟨Z_i Z_j⟩, then use those correlations to guide a cluster Monte Carlo — combining quantum insight with efficient classical post-processing.

**What you'll learn:**
1. How QAOA correlations differ from simple coupling constants
2. How correlation-guided cluster moves escape local minima
3. How Divi's QWC grouping reduces circuit count by up to 60%
4. How to run on QoroService for larger graphs (>18 qubits)

## 1. Setup & Backend Selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from algorithm import (
    generate_random_maxcut_graph,
    ising_energy,
    extract_qaoa_correlations,
    coupling_constant_correlations,
    correlation_guided_cluster_algorithm,
    simulated_annealing,
)

from divi.backends import ParallelSimulator, QoroService, JobConfig

### Choose Your Backend

For graphs with **≤18 nodes**, the local `ParallelSimulator` works well. For **larger graphs** (closer to the paper's 28-node and 100-node benchmarks), use `QoroService` to run on Qoro's cloud infrastructure.

Get your API key at [dash.qoroquantum.net](https://dash.qoroquantum.net).

In [ ]:
USE_CLOUD = False  # Set to True to use QoroService cloud backend
SHOTS = 10_000

if USE_CLOUD:
    # QoroService automatically reads QORO_API_KEY from your environment
    backend = QoroService(config=JobConfig(qpu_system="qoro_maestro", shots=SHOTS))
    print("☁️  Using QoroService cloud backend — can handle >18 qubit graphs")
else:
    backend = ParallelSimulator(shots=SHOTS)
    print("💻 Using local ParallelSimulator (best for ≤18 qubits)")

## 2. Generate a Frustrated Graph

We create a **random regular graph** with ±1 edge weights (Ising spin glass). Dense, frustrated graphs are hard for classical methods — exactly where quantum correlations should help.

| Parameter | Value | Why |
|-----------|-------|-----|
| `n_nodes` | 16 | Fits on local simulator (increase with QoroService) |
| `degree` | 12 | Dense graph → more frustration |
| Weights | ±1 random | Creates conflicting constraints |

> **Tip:** With `USE_CLOUD = True`, try `n_nodes=28` and `degree=10` to match the paper's primary benchmark.

In [ ]:
n_nodes = 16  # Increase to 28+ with QoroService
degree = 12
seed = 42

G = generate_random_maxcut_graph(n_nodes, degree, seed=seed)

n_pos = sum(1 for u, v, w in G.edges(data="weight") if w > 0)
n_neg = G.number_of_edges() - n_pos
print(f"Graph: {n_nodes} nodes, {degree}-regular, {G.number_of_edges()} edges ({n_pos}+, {n_neg}−)")

### Exact Ground State

For small graphs (≤22 nodes), we compute the exact ground state by brute force. This gives us the reference energy E₀ for computing approximation ratios.

In [ ]:
E_ground = np.inf
if n_nodes <= 22:
    for bits in range(2**n_nodes):
        config = np.array([1 - 2 * ((bits >> i) & 1) for i in range(n_nodes)])
        E = ising_energy(G, config)
        if E < E_ground:
            E_ground = E
    print(f"Exact ground state energy: E₀ = {E_ground:.1f}")
else:
    E_ground = None
    print(f"n={n_nodes} > 22: brute-force skipped (use results for relative comparison)")

## 3. Classical Baselines

### Simulated Annealing

Standard single-spin-flip SA — the default classical approach. On frustrated landscapes, it gets trapped in local minima.

In [ ]:
n_iterations_factor = 500
n_repetitions = 30

sa_result = simulated_annealing(
    G, n_iterations_factor=n_iterations_factor,
    n_repetitions=n_repetitions, seed=seed,
)

if E_ground is not None:
    sa_ratios = [e / E_ground for e in sa_result.energy_history]
    print(f"SA: Best E = {sa_result.best_energy:.1f}  |  Mean r = {np.mean(sa_ratios):.3f}")
else:
    print(f"SA: Best E = {sa_result.best_energy:.1f}")

### Coupling-Constant Clusters

Use the raw edge weights J_ij as "correlations" to guide cluster formation. This is the simplest baseline — no quantum information at all.

In [ ]:
Z_cc = coupling_constant_correlations(G)

cc_result = correlation_guided_cluster_algorithm(
    G, Z_cc, n_iterations_factor=n_iterations_factor,
    n_repetitions=n_repetitions, lambda_scale=1, seed=seed,
)

if E_ground is not None:
    cc_ratios = [e / E_ground for e in cc_result.energy_history]
    print(f"CC: Best E = {cc_result.best_energy:.1f}  |  Mean r = {np.mean(cc_ratios):.3f}  |  Accept = {cc_result.acceptance_rate:.1%}")
else:
    print(f"CC: Best E = {cc_result.best_energy:.1f}  |  Accept = {cc_result.acceptance_rate:.1%}")

## 4. QAOA Correlation Extraction

This is the **quantum step** — run QAOA on the Max-Cut Hamiltonian and extract two-point correlations ⟨Z_i Z_j⟩ from the optimized state.

Divi's **QWC (qubit-wise commuting) grouping** batches commuting ZZ observables into fewer measurement circuits — reducing the circuit count by up to 60%.

The `backend` parameter accepts `QoroService` for larger graphs — the correlation extraction is **identical** regardless of backend.

In [ ]:
qaoa_depths = [1, 2, 3]
lambda_scale = 4

qaoa_results = {}
qaoa_correlations = {"Coupling Constants": Z_cc}

for p in qaoa_depths:
    print(f"\n--- QAOA depth p={p} ---")
    Z_qaoa, qaoa_inst = extract_qaoa_correlations(
        G, n_layers=p, max_iterations=10, shots=SHOTS,
        backend=backend,  # Uses QoroService or ParallelSimulator
    )
    qaoa_correlations[f"QAOA p={p}"] = Z_qaoa

    qaoa_res = correlation_guided_cluster_algorithm(
        G, Z_qaoa, n_iterations_factor=n_iterations_factor,
        n_repetitions=n_repetitions, lambda_scale=lambda_scale, seed=seed,
    )
    qaoa_results[p] = qaoa_res

    if E_ground is not None:
        q_ratios = [e / E_ground for e in qaoa_res.energy_history]
        print(f"   Best E = {qaoa_res.best_energy:.1f}  |  Mean r = {np.mean(q_ratios):.3f}  |  "
              f"Accept = {qaoa_res.acceptance_rate:.1%}  |  Circuits = {qaoa_inst.total_circuit_count}")
    else:
        print(f"   Best E = {qaoa_res.best_energy:.1f}  |  "
              f"Accept = {qaoa_res.acceptance_rate:.1%}  |  Circuits = {qaoa_inst.total_circuit_count}")

## 5. Correlation Heatmaps

Visualize how correlations differ between coupling constants and QAOA at various depths. Look for increasing structure in the QAOA heatmaps.

In [ ]:
n_panels = len(qaoa_correlations)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4.5))

cmap = plt.cm.RdBu_r
vmax = max(np.abs(Z).max() for Z in qaoa_correlations.values())

for ax, (label, Z) in zip(axes, qaoa_correlations.items()):
    im = ax.imshow(Z, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="equal")
    ax.set_title(label, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Node j")
    ax.set_ylabel("Node i")

fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02, label="Correlation ⟨Z_i Z_j⟩")
fig.suptitle("Correlation Matrices: Coupling Constants vs QAOA", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Results Summary

Compare all methods. The approximation ratio $r = E / E_0$ measures how close each method gets to the exact ground state (1.0 = optimal).

In [ ]:
if E_ground is not None:
    print(f"{'Method':<28} {'Best E':>8} {'Mean r':>8} {'Best r':>8} {'σ(r)':>8} {'Accept':>8}")
    print(f"{'─'*68}")

    def print_row(name, result, show_accept=False):
        ratios = [e / E_ground for e in result.energy_history]
        mr = np.mean(ratios)
        sr = np.std(ratios)
        br = result.best_energy / E_ground
        acc = f"{result.acceptance_rate:>7.1%}" if show_accept else f"{'—':>8}"
        print(f"{name:<28} {result.best_energy:>8.1f} {mr:>8.3f} {br:>8.3f} {sr:>8.3f} {acc}")
else:
    print(f"{'Method':<28} {'Best E':>8} {'Mean E':>8} {'σ(E)':>8} {'Accept':>8}")
    print(f"{'─'*60}")

    def print_row(name, result, show_accept=False):
        me = np.mean(result.energy_history)
        se = np.std(result.energy_history)
        acc = f"{result.acceptance_rate:>7.1%}" if show_accept else f"{'—':>8}"
        print(f"{name:<28} {result.best_energy:>8.1f} {me:>8.1f} {se:>8.1f} {acc}")

print_row("Simulated Annealing", sa_result)
print_row("Cluster (Coupling Const.)", cc_result, show_accept=True)
for p, res in sorted(qaoa_results.items()):
    print_row(f"QAOA-Guided (p={p})", res, show_accept=True)

## 7. Energy Distribution

A box plot of energies across random restarts shows solution **reliability**, not just best-case performance.

In [ ]:
all_data = []
all_labels = []

all_data.append(sa_result.energy_history)
all_labels.append("SA")
all_data.append(cc_result.energy_history)
all_labels.append("CC")
for p in sorted(qaoa_results.keys()):
    all_data.append(qaoa_results[p].energy_history)
    all_labels.append(f"QAOA p={p}")

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(all_data, labels=all_labels, patch_artist=True)

colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6', '#1abc9c']
for patch, color in zip(bp['boxes'], colors[:len(all_data)]):
    patch.set_facecolor(color + '40')
    patch.set_edgecolor(color)

if E_ground is not None:
    ax.axhline(y=E_ground, color='red', linestyle='--', alpha=0.5, label=f'E₀ = {E_ground:.1f}')
    ax.legend()
ax.set_ylabel("Energy (lower = better)", fontweight="bold")
ax.set_title("Energy Distribution Across Random Restarts", fontweight="bold")
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Scaling Up with QoroService

To run on **larger graphs** matching the paper's benchmarks, set `USE_CLOUD = True` above and increase `n_nodes`:

```python
# Paper's primary benchmark: 28 nodes, 10-regular
n_nodes = 28
degree = 10
qaoa_depths = [1, 2, 3, 5]
```

QoroService handles the circuit execution transparently — your algorithm code stays **identical**.

## Reference

P. J. Eder et al., *Quantum-Guided Cluster Algorithms for Combinatorial Optimization*, arXiv:2508.10656 (2025).

See also: [AWS Quantum Technologies Blog Post](https://aws.amazon.com/blogs/quantum-computing/quantum-guided-cluster-algorithms-for-combinatorial-optimization/)